# Libaries


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer

# Optional: Visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Loading Data


In [ ]:
# Load your dataset
data = pd.read_csv('path/to/your_dataset.csv')

# Inspect the first few rows of the dataset
data.head()

# Text Preprocessing


In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

# Preprocess text function
stop_words = set(stopwords.words('english'))


def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove stop words
    text = ' '.join(word for word in text.split() if word not in stop_words)
    return text


# Apply preprocessing to the text column
data['processed_text'] = data['text'].apply(preprocess_text)

# Document Embeddings


In [ ]:
# Load Sentence-BERT model
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

# Generate embeddings for each sentence
data['embeddings'] = data['processed_text'].apply(lambda x: model.encode(x))

# Convert embeddings to a numpy array
X = np.vstack(data['embeddings'].values)

# Dimensionality Reduction


In [ ]:
# Reduce dimensions with PCA
pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X)

# Optional: Plot the reduced data
plt.figure(figsize=(10, 7))
plt.scatter(X_reduced[:, 0], X_reduced[:, 1], s=30, alpha=0.7)
plt.title('PCA-reduced Embeddings')
plt.show()

# Clustering


In [ ]:
# Initialize and fit K-Means with 2 clusters
kmeans = KMeans(n_clusters=2, random_state=42)
data['cluster'] = kmeans.fit_predict(X)

# Evaluate clustering with Silhouette Score
silhouette_avg = silhouette_score(X, data['cluster'])
print(f'Silhouette Score: {silhouette_avg}')

# Analyze Clusters


In [ ]:
# Inspect sample sentences from each cluster
for i in range(2):
    print(f"Cluster {i}:")
    print(data[data['cluster'] == i]['text'].sample(5).values)
    print("\n")

# Map clusters to labels (update based on interpretation)
cluster_to_label = {0: 'literal use', 1: 'non-literal use'}
data['use_type'] = data['cluster'].map(cluster_to_label)

# Visualization


In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(X_reduced[:, 0], X_reduced[:, 1],
                hue=data['use_type'], palette="deep", s=30)
plt.title('Clusters of Literal vs. Non-Literal Use')
plt.legend(loc='best')
plt.show()

## Save


In [ ]:
data[['text', 'verb', 'use_type']].to_csv('clustered_results.csv', index=False)